# Pareja C — Publicar un release (versión Kaggle)

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/11909/Poster2Movie-Clasificador/blob/main/notebooks/C_release.ipynb)

Lo mismo que `C_release.ipynb` (Colab), pero en Kaggle. Cuando un experimento (o un ensamble) mejora:
**el F1 macro en validación no baja y el promedio de los 4 débiles sube** contra el último release.
`release.py` mide en val **y en test** (la única vez que se mira test), copia el `.pt`, las métricas
y la receta a `releases/<tag>/` y escribe `LEEME.md`.

Versiones: PARCHE `v0.2.1` (arreglo sin cambiar el modelo), MENOR `v0.3.0` (modelo nuevo que
mejora), MAYOR `v1.0.0` (la que se entrega al repo principal). Un release no se reescribe.

Flujo: rama `release-<tag>` → Pull Request → al fusionarlo, la etiqueta (celda final) → GitHub
Actions publica el release.

**Diferencia con Colab:** Kaggle no puede montar Google Drive. Este notebook usa
[rclone](https://rclone.org/drive/) para **bajar** solo lo que necesita (la metadata del dataset, el
`.pt` del experimento y sus vectores, y los de la base para comparar). No sube nada al Drive: el
release va al repo. **CPU basta.**

## Antes de la primera vez

**En Kaggle** (panel derecho, *Settings*): *Accelerator* → **None** (CPU basta) e *Internet* →
**On** (pide verificar la cuenta con un teléfono).

**Llave del Drive (una vez por persona):** en tu computadora instala rclone
(<https://rclone.org/downloads/>) y corre:

```bash
rclone authorize "drive"
```

Se abre el navegador: entra con la cuenta de Google que tiene `poster2movie_datos` y
`poster2movie_clasificador` como acceso directo en *Mi unidad*. Al final la terminal imprime un texto
`{"access_token": ..., "refresh_token": ..., ...}`: cópialo completo (una línea, entre las llaves).

**En Kaggle**, *Add-ons → Secrets*:

| Secreto | Valor | ¿Obligatorio? |
| --- | --- | --- |
| `RCLONE_DRIVE_TOKEN` | el texto `{...}` de `rclone authorize` | sí |
| `GITHUB_TOKEN` | token *classic* con permiso `repo` (el mismo de Colab) | para subir `resultados/` sin que lo pida |
| `GIT_NOMBRE`, `GIT_CORREO` | tu nombre y correo de GitHub | igual |

Activa los secretos en **este** notebook (casilla junto a cada uno).

⚠️ El `RCLONE_DRIVE_TOKEN` da acceso a **todo tu Drive**: guárdalo solo en *Secrets* de Kaggle, nunca
en una celda, en el repo ni en el chat. Si se filtra, quítale el acceso a rclone en
<https://myaccount.google.com/permissions>.

Aquí no hace falta *Save & Run All*: tarda unos minutos y la etiqueta se pone a mano después del Pull Request.

## Preparar el entorno

Cambia `TAG`, `EXPERIMENTO` y `NOTAS`, y las carpetas si en tu *Mi unidad* los accesos directos
tienen otro nombre. Trabaja en la rama **`release-<tag>`** (se crea desde `main`).

In [ ]:
import json, os, subprocess, threading, time
from pathlib import Path

# --- Qué se publica ---
TAG = "v0.2.0"                 # el siguiente número
EXPERIMENTO = "C7_todos"       # el candidato que salió de C_cierre
NOTAS = "Qué cambió y por qué ayudó (va en el LEEME del release)"
ANTERIOR = "v0.1.0"            # el último release (contra el que se compara)

# --- Drive del equipo (rutas dentro de *Mi unidad*) ---
REMOTO = "gdrive"                          # nombre interno de rclone, no hay que cambiarlo
DRIVE_DATOS = "poster2movie_datos"          # dataset compartido (solo se LEE)
DRIVE_SALIDAS = "poster2movie_clasificador"  # carpeta del equipo (aquí se SUBEN vectores, modelos, predicciones)

# --- Repo ---
REPO = "11909/Poster2Movie-Clasificador"
RAMA = f"release-{TAG}"
DESTINO = "/kaggle/working/clasificador"   # en /kaggle/working: queda en la salida del notebook

# --- Carpetas locales de Kaggle ---
os.environ["P2M_DATOS"] = "/tmp/p2m_datos"                  # copia local del dataset (no va a la salida)
os.environ["P2M_SALIDAS"] = "/kaggle/working/p2m_salidas"   # se sincroniza con DRIVE_SALIDAS
for carpeta in (os.environ["P2M_DATOS"], os.environ["P2M_SALIDAS"]):
    Path(carpeta).mkdir(parents=True, exist_ok=True)
DATOS, SALIDAS = os.environ["P2M_DATOS"], os.environ["P2M_SALIDAS"]
DRIVE = f"{REMOTO}:{DRIVE_SALIDAS}"
EN_SEGUNDO_PLANO = os.environ.get("KAGGLE_KERNEL_RUN_TYPE") == "Batch"   # "Save & Run All"


def secreto(nombre):
    # Un secreto de Kaggle (Add-ons -> Secrets), o None si no existe o no está activado.
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(nombre)
    except Exception:
        return None


# --- rclone: instalar y configurar con la llave del Drive ---
if not Path("/usr/local/bin/rclone").exists():
    subprocess.run("curl -sSL -o /tmp/rclone.zip https://downloads.rclone.org/rclone-current-linux-amd64.zip"
                   " && unzip -q -o /tmp/rclone.zip -d /tmp/rclone"
                   " && cp /tmp/rclone/rclone-*-linux-amd64/rclone /usr/local/bin/", shell=True, check=True)

token = secreto("RCLONE_DRIVE_TOKEN")
if not token:
    raise RuntimeError("Falta el secreto RCLONE_DRIVE_TOKEN (ver 'Antes de la primera vez'): "
                       "Add-ons -> Secrets, y activarlo para este notebook.")
json.loads(token)  # si falla aquí, el secreto no es el texto {...} completo de `rclone authorize "drive"`
conf = Path.home() / ".config/rclone/rclone.conf"
conf.parent.mkdir(parents=True, exist_ok=True)
conf.write_text(f"[{REMOTO}]\ntype = drive\nscope = drive\ntoken = {token.strip()}\n")
conf.chmod(0o600)
del token


def rclone(*args, check=True):
    return subprocess.run(["rclone", *args], check=check, capture_output=True, text=True)


for carpeta in (DRIVE_DATOS, DRIVE_SALIDAS):
    r = rclone("lsf", f"{REMOTO}:{carpeta}", "--max-depth", "1", check=False)
    if r.returncode != 0:
        raise RuntimeError(f"No encuentro '{carpeta}' en tu Mi unidad. ¿Agregaste el acceso directo? "
                           f"Detalle de rclone:\n{r.stderr}")
print("Drive conectado:", DRIVE_DATOS, "y", DRIVE_SALIDAS)

def bajar_vectores(nombres):
    # Trae del Drive los .npz (y su .json) de cada vector; los que ya estén aquí no se vuelven a bajar.
    # Orden de búsqueda igual que `rutas.ruta_vectores`: carpeta del equipo, dataset compartido, alias de la base.
    for nombre in nombres:
        destino = Path(SALIDAS) / "vectores"
        if (destino / f"{nombre}.npz").exists():
            continue
        if rclone("lsf", f"{DRIVE}/vectores/{nombre}.npz", check=False).stdout.strip():
            rclone("copy", f"{DRIVE}/vectores", str(destino), "--include", f"/{nombre}.*")
        elif rclone("lsf", f"{REMOTO}:{DRIVE_DATOS}/vectores/{nombre}.npz", check=False).stdout.strip():
            rclone("copy", f"{REMOTO}:{DRIVE_DATOS}/vectores", f"{DATOS}/vectores", "--include", f"/{nombre}.*")
        elif nombre == "clip_b32__centro":   # los vectores de la base viven en el dataset compartido
            rclone("copy", f"{REMOTO}:{DRIVE_DATOS}", DATOS, "--include", "/embeddings_clip_b32.npz")
        else:
            print(f"Aviso: no encuentro los vectores '{nombre}' en el Drive.")
            continue
        print("Bajado:", nombre)


In [ ]:
# --- Repo y dependencias ---
if not os.path.isdir(DESTINO):
    !git clone -q --recurse-submodules https://github.com/{REPO}.git {DESTINO}
%cd {DESTINO}
!git fetch -q origin
!git checkout -q main && git pull -q origin main
!git checkout -q -b {RAMA} 2>/dev/null || git checkout -q {RAMA}
!git submodule update -q --init
!pip install -q -r requirements.txt
!nvidia-smi -L 2>/dev/null || echo "Sin GPU (basta para todo salvo los pasos que la piden)"

## Bajar del Drive

La metadata del dataset, el `.pt` del experimento y sus vectores (si es un ensamble o apilado, los de
todos sus miembros), y los vectores de la base para la comparación.

In [ ]:
import torch
print("Bajando la metadata del dataset...")
rclone("copy", f"{REMOTO}:{DRIVE_DATOS}", DATOS, "--include", "/metadata.parquet")
rclone("copy", f"{DRIVE}/modelos", f"{SALIDAS}/modelos", "--include", f"/{EXPERIMENTO}.pt")
ruta_pt = Path(SALIDAS, "modelos", f"{EXPERIMENTO}.pt")
if not ruta_pt.exists():
    raise FileNotFoundError(f"No encuentro modelos/{EXPERIMENTO}.pt en {DRIVE_SALIDAS}: ¿se entrenó y se subió?")
guardado = torch.load(ruta_pt, map_location="cpu", weights_only=False)
VECTORES = list(guardado.get("vectores") or ["clip_b32__centro"])
print(f"{EXPERIMENTO} usa: {VECTORES}")
bajar_vectores(dict.fromkeys(VECTORES + ["clip_b32__centro"]))

## Comparar contra el último release (validación) y preparar el release

In [ ]:
!python -m clasificador.comparar {ANTERIOR} {EXPERIMENTO}   # contra el último release, en validación

In [ ]:
!python -m clasificador.release {TAG} {EXPERIMENTO} --notas "{NOTAS}"
print(open(f"releases/{TAG}/LEEME.md", encoding="utf-8").read())

## Subir a la rama `release-<tag>`

Se sube a **`release-<tag>`**, nunca a `main`; luego abran el Pull Request (lo revisa alguien de
**otra** pareja). Con los secretos `GITHUB_TOKEN`, `GIT_NOMBRE` y `GIT_CORREO` no pregunta nada (sirve
en *Save & Run All*); sin ellos los pide aquí. Si corrió en segundo plano sin secretos, los resultados
quedan en la salida del notebook (`clasificador/releases/`).

In [ ]:
from getpass import getpass

token = secreto("GITHUB_TOKEN")
nombre_git, correo_git = secreto("GIT_NOMBRE"), secreto("GIT_CORREO")
if EN_SEGUNDO_PLANO and not (token and nombre_git and correo_git):
    print("Sin GITHUB_TOKEN / GIT_NOMBRE / GIT_CORREO en segundo plano: no se sube a GitHub. "
          "El release está en la salida del notebook (clasificador/releases/).")
else:
    nombre_git = nombre_git or input("Tu nombre: ")
    correo_git = correo_git or input("Tu correo de GitHub: ")
    mensaje = f"Release {TAG} (desde Kaggle)" if EN_SEGUNDO_PLANO \
        else input("Qué hiciste (mensaje del commit): ")
    token = token or getpass("Token de GitHub: ")
    !git config user.name "{nombre_git}"
    !git config user.email "{correo_git}"
    !git add releases resultados
    !git commit -q -m "{RAMA}: {mensaje}"
    !git push -q https://{token}@github.com/{REPO}.git HEAD:{RAMA}
    print(f"Abre el Pull Request: https://github.com/{REPO}/compare/main...{RAMA}")
del token

## Después de fusionar el Pull Request: la etiqueta

Al subir la etiqueta, GitHub Actions crea el release con los archivos de `releases/<tag>/`. Corre esta
celda **solo después** de fusionar el Pull Request en `main`.

In [ ]:
from getpass import getpass
!git fetch -q origin && git checkout -q main && git pull -q origin main
if not Path(f"releases/{TAG}/LEEME.md").exists():
    raise SystemExit(f"releases/{TAG} todavía no está en main: fusiona primero el Pull Request.")
!git tag {TAG}
token = secreto("GITHUB_TOKEN") or getpass("Token de GitHub: ")
!git push -q https://{token}@github.com/{REPO}.git {TAG}
del token
print(f"https://github.com/{REPO}/releases/tag/{TAG}")

## Probar con un cartel

En Kaggle no hay botón para subir una imagen desde la celda: súbela como *Dataset* (*Add Input*) y
pon su ruta abajo, o prueba el release en `C_release` (Colab).

In [ ]:
from PIL import Image
from clasificador.predictor import Clasificador
from p2m.config import GENEROS_ES

CARTEL = "/kaggle/input/<tu-dataset>/cartel.jpg"   # cámbialo
if Path(CARTEL).exists():
    clasificador = Clasificador(ruta_pt)
    resultado = clasificador.predecir(Image.open(CARTEL).convert("RGB"))
    print(", ".join(GENEROS_ES[g] for g in resultado["generos"]))
    top = sorted(resultado["probabilidades"].items(), key=lambda x: -x[1])[:5]
    print(", ".join(f"{GENEROS_ES[g]} {p:.2f}" for g, p in top))
else:
    print("Pon en CARTEL la ruta de una imagen.")